# Module 6 Homework – Yellow Taxi 2025-11

**Data**: `yellow_tripdata_2025-11.parquet` (pickup: `tpep_pickup_datetime`)

Windows: Run Jupyter via `start_jupyter_with_hadoop.bat` or set HADOOP_HOME before Spark.

In [1]:
# Optional: download data if not present
# Run in terminal: wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
# Run in terminal: wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

import os
import urllib.request

parquet_path = 'yellow_tripdata_2025-11.parquet'
csv_path = 'taxi_zone_lookup.csv'

if not os.path.exists(parquet_path):
    urllib.request.urlretrieve('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet', parquet_path)
    print(f'Downloaded: {parquet_path}')
if not os.path.exists(csv_path):
    urllib.request.urlretrieve('https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv', csv_path)
    print(f'Downloaded: {csv_path}')

In [2]:
# Windows: Set HADOOP_HOME before Spark (run first if using Cursor/Anaconda)
import os
_hadoop_dirs = [
    r"E:\IT_SPACES\AI\ZoomCamp\DE\06\tools\hadoop-3.3.5",
    r"E:\IT_SPACES\AI\ZoomCamp\DE\tools\hadoop-3.3.5",
]
for _d in _hadoop_dirs:
    if os.path.isdir(_d):
        os.environ["HADOOP_HOME"] = _d
        os.environ["PATH"] = os.environ.get("PATH", "") + os.pathsep + os.path.join(_d, "bin")
        print("HADOOP_HOME:", _d)
        break
else:
    print("Warning: hadoop folder not found. Run start_jupyter_with_hadoop.bat")

HADOOP_HOME: E:\IT_SPACES\AI\ZoomCamp\DE\06\tools\hadoop-3.3.5


In [3]:
import pyspark  # Spark 기능을 파이썬 환경으로 불러옴
from pyspark.sql import SparkSession  # Spark 엔진을 제어하는 핵심 세션 관리 도구를 가져옴
from pyspark.sql import types  # 데이터 프레임의 컬럼 타입을 지정하는 도구들을 가져옴

In [4]:
# 4. Spark 세션을 생성하기 위한 빌더(Builder)를 시작합니다.
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

    # Spark 세션을 만들기 위한 설정 시작
    # .master("local[*]"): 배치 파일이 열어준 환경 위에서 내 컴퓨터의 모든 CPU 코어를 다 쓰겠다고 선언합니다.
                            # 내 컴퓨터의 모든 CPU 코어를 다 써서 병렬 처리하도록 설정
    # .appName('test'): 관리자 화면(Spark UI)에 표시될 작업의 이름을 'test'로 정합니다.
                        # 실행 중인 작업의 이름을 'test'로 지정 (Spark UI 식별용)
    # .getOrCreate(): 이미 실행 중인 세션이 있으면 그걸 쓰고, 없으면 새로 만듭니다.
                      # 설정대로 세션을 새로 생성하거나 이미 있으면 연결함

In [5]:
# 5. 현재 연결된 Spark 엔진의 버전 정보를 요청합니다.
# 배치 파일이 주입한 HADOOP_HOME 덕분에 에러 없이 버전 정보를 응답받습니다.
spark.version  # 현재 생성된 Spark 엔진의 버전 정보를 확인 

'3.5.0'

## Q2: Repartition (4) and write Parquet

In [6]:
# 1. 지정된 경로의 Parquet 파일을 읽어와서 데이터프레임(df) 형태로 저장함
df = spark.read.parquet('yellow_tripdata_2025-11.parquet') 

# 2. 불러온 데이터의 컬럼명과 데이터 타입(스키마) 구조를 트리 형태로 출력함
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [7]:
# 1. 데이터를 4개의 파티션(파일 단위)으로 다시 나눔 
df = df.repartition(4) 

# 2. 지정된 경로에 데이터를 저장함
df.write.mode('overwrite').parquet('data/pq/yellow/2025/11/') 
    # .mode('overwrite'): 같은 경로에 데이터가 이미 있다면 덮어쓰기
    # .parquet(...): 지정한 폴더 안에 4개의 파티션된 파일을 생성함

In [8]:
# 1. 지정된 폴더 경로에 있는 모든 파티션(4개 파일)을 하나의 데이터프레임으로 읽어옴
df = spark.read.parquet('data/pq/yellow/2025/11/')

In [9]:
import os # 파일 시스템 작업을 위한 라이브러리 임포트

# 1. 대상 폴더 경로 설정
folder_path = 'data/pq/yellow/2025/11/'

# 2. .parquet 확장자로 끝나는 파일들만 리스트로 필터링
parquet_files = [f for f in os.listdir(folder_path) if f.endswith('.parquet')]

# 3. 개별 파일 정보 출력 및 전체 크기 합산
total_size = 0
print("--- Individual File Details ---")
for file_name in parquet_files:
    file_path = os.path.join(folder_path, file_name) # 파일의 전체 경로 생성
    file_size_bytes = os.path.getsize(file_path)     # 파일 크기 추출 (Byte 단위)
    file_size_mb = file_size_bytes / (1024 * 1024)   # MB 단위로 변환
    total_size += file_size_bytes                    # 전체 크기에 합산
    print(f"File name: {file_name} | Size: {file_size_mb:.2f} MB") # 파일명과 크기 출력

# 4. 파일 개수와 평균 크기 계산 결과 출력
print("-" * 30)
if len(parquet_files) > 0:
    avg_size_bytes = total_size / len(parquet_files) # 평균 바이트 계산
    avg_size_mb = avg_size_bytes / (1024 * 1024)     # 평균 MB 변환
    
    print(f"Total number of files: {len(parquet_files)}") # 총 파일 개수 출력
    print(f"Average file size: {avg_size_mb:.2f} MB")     # 최종 평균 크기 출력
else:
    print("No .parquet files found in the specified path.") # 파일이 없을 경우 메시지

--- Individual File Details ---
File name: part-00000-6b4f440b-2cea-4122-b0d2-73517c8a8b1c-c000.snappy.parquet | Size: 24.40 MB
File name: part-00001-6b4f440b-2cea-4122-b0d2-73517c8a8b1c-c000.snappy.parquet | Size: 24.39 MB
File name: part-00002-6b4f440b-2cea-4122-b0d2-73517c8a8b1c-c000.snappy.parquet | Size: 24.42 MB
File name: part-00003-6b4f440b-2cea-4122-b0d2-73517c8a8b1c-c000.snappy.parquet | Size: 24.42 MB
------------------------------
Total number of files: 4
Average file size: 24.41 MB


## Q3: How many taxi trips were there on November 15?

In [10]:
from pyspark.sql import functions as F  # 데이터 프레임 조작에 필요한 다양한 내장 함수(컬럼 계산, 필터링 등)를 제공하는 모듈을 F라는 별칭으로 가져옴

In [11]:
# 1. tpep_pickup_datetime 컬럼에서 날짜 정보만 추출하여 'pickup_date' 컬럼을 새로 생성
# 2. 승차 날짜가 '2025-11-15'인 데이터만 필터링
# 3. 해당 조건에 맞는 행의 총 개수를 계산
df \
    .withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .filter("pickup_date = '2025-11-15'") \
    .count()

162604

In [12]:
df.createOrReplaceTempView('yellow_2025_11') # 데이터프레임을 'yellow_2025_11'이라는 이름의 임시 뷰(테이블)로 등록하여 SQL 쿼리를 사용할 수 있게 함

spark.sql("""
SELECT COUNT(1)
FROM yellow_2025_11
WHERE to_date(tpep_pickup_datetime) = '2025-11-15'
""").show() # SQL 문을 실행하여 2025년 11월 15일자 승차 기록의 총 개수를 구하고 결과를 화면에 출력함

+--------+
|count(1)|
+--------+
|  162604|
+--------+



## Q4: Longest trip for each day (duration in hours)

In [13]:
# 1. unix_timestamp 함수를 사용하여 시간 차이를 초(seconds) 단위로 계산
# 2. 초 단위 값을 3600으로 나눠 시간(hours) 단위 컬럼 'duration_hours' 생성
# 3. 승차 일시에서 날짜만 추출하여 'pickup_date' 컬럼 생성
# 4. 날짜별로 그룹화하여 가장 긴 운행 시간(max)을 구함
# 5. 최대 운행 시간 기준 내림차순 정렬 후 상위 5개 출력
df \
    .withColumn('duration_sec', F.unix_timestamp(df.tpep_dropoff_datetime) - F.unix_timestamp(df.tpep_pickup_datetime)) \
    .withColumn('duration_hours', F.col('duration_sec') / 3600) \
    .withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .groupBy('pickup_date') \
    .max('duration_hours') \
    .orderBy(F.col('max(duration_hours)').desc()) \
    .limit(5) \
    .show()

+-----------+-------------------+
|pickup_date|max(duration_hours)|
+-----------+-------------------+
| 2025-11-26|  90.64666666666666|
| 2025-11-27|  76.94833333333334|
| 2025-11-03|  76.21388888888889|
| 2025-11-07|  69.28861111111111|
| 2025-11-18|  67.08055555555555|
+-----------+-------------------+



In [14]:
# 1. unix_timestamp를 사용하여 두 시간 사이의 차이를 초 단위로 구함
# 2. 초 단위를 3600.0으로 나누어 시간(hours) 단위로 환산
# 3. 'pickup_date'별로 그룹화하여 최대 운행 시간(MAX)을 계산
# 4. 가장 긴 운행 시간 순으로 내림차순 정렬하여 상위 10개 출력
spark.sql("""
SELECT
    to_date(tpep_pickup_datetime) AS pickup_date,
    MAX(unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600.0 AS duration_hours
FROM yellow_2025_11
GROUP BY 1
ORDER BY 2 DESC
LIMIT 10
""").show()

+-----------+--------------+
|pickup_date|duration_hours|
+-----------+--------------+
| 2025-11-26|     90.646667|
| 2025-11-27|     76.948333|
| 2025-11-03|     76.213889|
| 2025-11-07|     69.288611|
| 2025-11-18|     67.080556|
| 2025-11-22|     63.368333|
| 2025-11-01|     56.382222|
| 2025-11-05|     42.720556|
| 2025-11-06|     41.614444|
| 2025-11-24|     38.074444|
+-----------+--------------+



## Q5: Most frequent VendorID

In [15]:
# VendorID별로 데이터를 그룹화하여 각 공급업체별 전체 운행 건수를 계산함
# 계산된 운행 건수(COUNT)를 기준으로 내림차순 정렬하여 가장 많은 건수부터 표시함
# 상위 5개의 결과만 화면에 출력함
spark.sql("""
SELECT VendorID, COUNT(1)
FROM yellow_2025_11
GROUP BY 1
ORDER BY 2 DESC
LIMIT 5
""").show()

+--------+--------+
|VendorID|count(1)|
+--------+--------+
|       2| 3295835|
|       1|  821333|
|       7|   60043|
|       6|    4233|
+--------+--------+



In [16]:
# VendorID별로 데이터를 그룹화하여 각 업체별 총 운행 건수를 집계함
# 집계된 'count' 컬럼을 기준으로 내림차순(건수가 많은 순서) 정렬함
# 상위 5개의 결과만 선택함
# 최종 결과를 화면에 출력함
df.groupBy('VendorID').count() \
    .orderBy(F.col('count').desc()) \
    .limit(5) \
    .show()

+--------+-------+
|VendorID|  count|
+--------+-------+
|       2|3295835|
|       1| 821333|
|       7|  60043|
|       6|   4233|
+--------+-------+



## Q6: Most common pickup/dropoff zone pair

In [17]:
# taxi_zone_lookup.csv 파일을 읽어서 df_zones 데이터프레임 생성
# header=true: 첫 번째 행을 컬럼명으로 사용
df_zones = spark.read \
    .option('header', 'true') \
    .csv('taxi_zone_lookup.csv') \
    .withColumn('LocationID', F.col('LocationID').cast('integer')) 
    # .withColumn: 'LocationID' 컬럼의 데이터 타입을 문자열에서 정수형(integer)으로 변환
    # (조인 연산 시 성능 향상 및 데이터 무결성을 위해 형변환 수행)

# 데이터프레임을 'zones'라는 이름의 임시 뷰(Temporary View)로 등록
# 이제 spark.sql("SELECT * FROM zones")와 같이 SQL 문법을 사용하여 쿼리 실행 가능
df_zones.createOrReplaceTempView('zones')

In [18]:
spark.sql("""
SELECT
    CONCAT(pul.Zone, ' / ', dol.Zone) AS pu_do_pair,
    COUNT(1)
FROM yellow_2025_11 y
LEFT JOIN zones pul ON y.PULocationID = pul.LocationID
LEFT JOIN zones dol  ON y.DOLocationID = dol.LocationID
GROUP BY 1
ORDER BY 2 DESC
LIMIT 5
""").show()

+--------------------+--------+
|          pu_do_pair|count(1)|
+--------------------+--------+
|Upper East Side S...|   28326|
|Upper East Side N...|   24720|
|Upper East Side S...|   20512|
|Upper East Side N...|   17820|
|Midtown Center / ...|   13444|
+--------------------+--------+



In [19]:
# 가장 빈도가 낮은(Least frequent) 승차 구역(PULocationID) 상위 10개 추출
spark.sql("""
SELECT
    pul.Zone,
    COUNT(1) AS count
FROM yellow_2025_11 y
LEFT JOIN zones pul ON y.PULocationID = pul.LocationID
GROUP BY 1
ORDER BY 2 ASC
LIMIT 10
""").show()

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|       Arden Heights|    1|
|Eltingville/Annad...|    1|
|Governor's Island...|    1|
|       Port Richmond|    3|
|       Rikers Island|    4|
|   Rossville/Woodrow|    4|
|         Great Kills|    4|
| Green-Wood Cemetery|    4|
|         Jamaica Bay|    5|
|         Westerleigh|   12|
+--------------------+-----+

